# Batch Processing — Fraud Detection Pipeline

This notebook:
1. Starts a local Spark session
2. Loads all five raw dataset files
3. Cleans and joins the data
4. Engineers new features
5. Runs aggregations
6. Saves outputs to `/processed`

In [1]:
# Se importan os para manejar rutas del sistema y json para leer archivos JSON
import os
import json

# ─── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR      = "D:/UP/BigData/Pryecto1"
RAW_DIR       = "D:/UP/BigData/Pryecto1/raw"
PROCESSED_DIR = "D:/UP/BigData/Pryecto1/processed"

# Se crea la carpeta processed si no existe para no causar errores al guardar resultados
os.makedirs(PROCESSED_DIR, exist_ok=True)
print(f"Base dir      : {BASE_DIR}")
print(f"Raw dir       : {RAW_DIR}")
print(f"Processed dir : {PROCESSED_DIR}")

Base dir      : D:/UP/BigData/Pryecto1
Raw dir       : D:/UP/BigData/Pryecto1/raw
Processed dir : D:/UP/BigData/Pryecto1/processed


## 1 — Start Spark Session

In [2]:
import os
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] += r";C:\hadoop\bin"

# Se importa SparkSession, que es el punto de entrada para cualquier programa en PySpark
from pyspark.sql import SparkSession

print("Starting Spark session...")

# Se configura y crea la sesión de Spark con los parámetros del proyecto.
# .master("local[*]") indica que Spark debe usar todos los núcleos del CPU disponibles de forma local.
# spark.driver.memory limita la memoria RAM que usa el proceso principal.
# spark.sql.shuffle.partitions controla cuántas particiones se crean en operaciones como join o groupBy.
# La política LEGACY en timeParserPolicy permite parsear fechas con formatos más antiguos sin errores.
spark = (
    SparkSession.builder
    .appName("FraudDetection_BatchProcessing")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.ansi.enabled", "false")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .config("spark.local.dir", "D:/spark-temp")
    .config("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2")  # ← agregar
    .getOrCreate()
)



# Se suprime la verbosidad del log de Spark para mostrar solo advertencias y errores
spark.sparkContext.setLogLevel("WARN")
print(f"✓ Spark session started — version {spark.version}")
print(f"  Master : {spark.sparkContext.master}")
print(f"  App ID : {spark.sparkContext.applicationId}")

Starting Spark session...
✓ Spark session started — version 4.1.1
  Master : local[*]
  App ID : local-1776097521781


## 2 — Load Raw Files into Spark DataFrames

In [3]:
# Se importan las funciones de transformación de columnas y el tipo FloatType para conversiones
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType, TimestampType

# ── transactions ──────────────────────────────────────────────────────────────
# Se lee el CSV de transacciones con inferencia automática de tipos (inferSchema=True).
# header=True indica que la primera fila contiene los nombres de las columnas.
transactions_df = spark.read.csv(
    r"D:\UP\BigData\Pryecto1\raw\transactions_data.csv",
    header=True, inferSchema=True,
    timestampFormat="yyyy-MM-dd HH:mm:ss"
)

print(f"  Rows    : {transactions_df.count():,}")
print(f"  Columns : {transactions_df.columns}")

# ── cards ─────────────────────────────────────────────────────────────────────
# Se carga el dataset de tarjetas bancarias con la misma configuración
print("\nLoading cards_data.csv ...")
cards_df = spark.read.csv(
    os.path.join(RAW_DIR, "cards_data.csv"),
    header=True, inferSchema=True
)
print(f"  Rows    : {cards_df.count():,}")
print(f"  Columns : {cards_df.columns}")

# ── users ─────────────────────────────────────────────────────────────────────
# Se carga el dataset de usuarios que contiene información demográfica
print("\nLoading users_data.csv ...")
users_df = spark.read.csv(
    os.path.join(RAW_DIR, "users_data.csv"),
    header=True, inferSchema=True
)
print(f"  Rows    : {users_df.count():,}")
print(f"  Columns : {users_df.columns}")

  Rows    : 13,305,915
  Columns : ['id', 'date', 'client_id', 'card_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors']

Loading cards_data.csv ...
  Rows    : 6,146
  Columns : ['id', 'client_id', 'card_brand', 'card_type', 'card_number', 'expires', 'cvv', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date', 'year_pin_last_changed', 'card_on_dark_web']

Loading users_data.csv ...
  Rows    : 2,000
  Columns : ['id', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender', 'address', 'latitude', 'longitude', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards']


In [4]:
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# ── MCC codes (JSON) ──────────────────────────────────────────────────────────
print("Loading mcc_codes.json ...")

with open(os.path.join(RAW_DIR, "mcc_codes.json"), "r", encoding="utf-8") as f:
    mcc_raw = json.load(f)

if isinstance(mcc_raw, dict):
    mcc_pd = pd.DataFrame(list(mcc_raw.items()), columns=["mcc_code", "description"])
else:
    mcc_pd = pd.DataFrame(mcc_raw)
    for c in mcc_pd.columns:
        if "mcc" in c.lower() or "code" in c.lower():
            mcc_pd = mcc_pd.rename(columns={c: "mcc_code"})
            break

mcc_pd["mcc_code"]    = mcc_pd["mcc_code"].astype(str)
mcc_pd["description"] = mcc_pd["description"].astype(str)

_mcc_tmp = RAW_DIR.replace("\\", "/") + "/_tmp_mcc.csv"
mcc_pd.to_csv(_mcc_tmp, index=False)

mcc_schema = StructType([
    StructField("mcc_code",    StringType(), True),
    StructField("description", StringType(), True),
])
mcc_df = spark.read.csv(_mcc_tmp, header=True, schema=mcc_schema)

print(f"  Rows    : {mcc_df.count():,}")
print(f"  Columns : {mcc_df.columns}")

# ── Fraud labels (JSON) ───────────────────────────────────────────────────────
print("Loading train_fraud_labels.json ...")

with open(os.path.join(RAW_DIR, "train_fraud_labels.json"), "r", encoding="utf-8") as f:
    labels_raw = json.load(f)

if isinstance(labels_raw, dict):
    first_val = next(iter(labels_raw.values()))
    if isinstance(first_val, dict):
        labels_raw = first_val

if isinstance(labels_raw, dict):
    labels_pd = pd.DataFrame(list(labels_raw.items()), columns=["transaction_id", "is_fraud"])
else:
    labels_pd = pd.DataFrame(labels_raw)
    for c in labels_pd.columns:
        if "id" in c.lower() and "fraud" not in c.lower():
            labels_pd = labels_pd.rename(columns={c: "transaction_id"})
        if "fraud" in c.lower() or "label" in c.lower():
            labels_pd = labels_pd.rename(columns={c: "is_fraud"})

labels_pd["transaction_id"] = labels_pd["transaction_id"].astype(str)
labels_pd["is_fraud"] = labels_pd["is_fraud"].map(
    lambda x: 1 if str(x).strip().lower() in ("1", "yes", "true") else 0
)

_labels_tmp = RAW_DIR.replace("\\", "/") + "/_tmp_labels.csv"
labels_pd.to_csv(_labels_tmp, index=False)

labels_schema = StructType([
    StructField("transaction_id", StringType(),  True),
    StructField("is_fraud",       IntegerType(), True),
])
labels_df = spark.read.csv(_labels_tmp, header=True, schema=labels_schema)

print(f"  Rows    : {labels_df.count():,}")
print(f"  Columns : {labels_df.columns}")


Loading mcc_codes.json ...
  Rows    : 0
  Columns : ['mcc_code', 'description']
Loading train_fraud_labels.json ...
  Rows    : 0
  Columns : ['transaction_id', 'is_fraud']


## 3 — Clean Transactions DataFrame

In [5]:
print("=" * 60)
print("STEP 3 — Cleaning transactions DataFrame")
print("=" * 60)

# Se detectan dinámicamente los nombres de las columnas de monto y timestamp
t_cols    = [c.lower() for c in transactions_df.columns]
orig_cols = transactions_df.columns

amount_col = next(
    (orig_cols[i] for i, c in enumerate(t_cols) if c == "amount"), None
)
ts_col = next(
    (orig_cols[i] for i, c in enumerate(t_cols)
     if c in ("timestamp", "date", "trans_date", "transaction_date")), None
)

print(f"  Detected amount column    : {amount_col}")
print(f"  Detected timestamp column : {ts_col}")

raw_count = transactions_df.count()
print(f"  Rows before cleaning      : {raw_count:,}")

drop_null_cols = [c for c in [amount_col, ts_col] if c]
clean_df = transactions_df.dropna(subset=drop_null_cols)
clean_df = clean_df.dropDuplicates()

if amount_col:
    clean_df = clean_df.withColumn(
        amount_col,
        F.regexp_replace(F.col(amount_col).cast("string"), r'[\$,]', '').cast(FloatType())
    )

if ts_col:
    clean_df = clean_df.withColumn(
        ts_col,
        F.coalesce(
            F.to_timestamp(F.col(ts_col), "yyyy-MM-dd HH:mm:ss"),
            F.to_timestamp(F.col(ts_col), "yyyy-MM-dd"),
            F.to_timestamp(F.col(ts_col))
        )
    )

# Estandarizar nombres de columnas a minúsculas
for old_col in clean_df.columns:
    new_col = old_col.lower().replace(" ", "_").replace("-", "_")
    if old_col != new_col:
        clean_df = clean_df.withColumnRenamed(old_col, new_col)

for df_name, df_var in [("cards_df", cards_df), ("users_df", users_df)]:
    new_df = df_var
    for old_col in df_var.columns:
        new_col = old_col.lower().replace(" ", "_").replace("-", "_")
        if old_col != new_col:
            new_df = new_df.withColumnRenamed(old_col, new_col)
    if df_name == "cards_df":
        cards_df = new_df
    else:
        users_df = new_df

clean_count = clean_df.count()
print(f"  Rows after cleaning       : {clean_count:,}")
print(f"  Rows dropped              : {raw_count - clean_count:,}")
clean_df.printSchema()


STEP 3 — Cleaning transactions DataFrame
  Detected amount column    : amount
  Detected timestamp column : date
  Rows before cleaning      : 13,305,915
  Rows after cleaning       : 13,305,915
  Rows dropped              : 0
root
 |-- id: integer (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- card_id: integer (nullable = true)
 |-- amount: float (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: double (nullable = true)
 |-- mcc: integer (nullable = true)
 |-- errors: string (nullable = true)



## 4 — Join All DataFrames

In [6]:
print("=" * 60)
print("STEP 4 — Joining DataFrames")
print("=" * 60)

card_key = next((c for c in clean_df.columns if "card_id" in c), "card_id")
user_key  = next((c for c in clean_df.columns if "user_id" in c or "client_id" in c), "user_id")

mcc_key_txn = next(
    (c for c in clean_df.columns if "mcc" in c),
    next((c for c in clean_df.columns if "merchant_category" in c or "category" in c), None)
)

# Clave de usuario en users_df (puede llamarse distinto que en clean_df)
users_id_key = next(
    (c for c in users_df.columns if "user_id" in c or "client_id" in c or c == user_key),
    "user_id"
)

print(f"  Join: transactions ⟕ cards   on {card_key}")
print(f"        result       ⟕ users   on {user_key} == {users_id_key}")
print(f"        result       ⟕ mcc     on {mcc_key_txn}")

cards_cols_to_rename = [c for c in cards_df.columns if c != card_key and c in clean_df.columns]
for c in cards_cols_to_rename:
    cards_df = cards_df.withColumnRenamed(c, f"card_{c}")

users_cols_to_rename = [c for c in users_df.columns if c != users_id_key and c in clean_df.columns]
for c in users_cols_to_rename:
    users_df = users_df.withColumnRenamed(c, f"user_{c}")

# 1. Transacciones ⟕ tarjetas
joined = clean_df.join(cards_df, on=card_key, how="left")
print(f"  After txn ⟕ cards  : {joined.count():,} rows")

# 2. Resultado ⟕ usuarios (maneja nombres distintos entre DataFrames)
if user_key == users_id_key:
    joined = joined.join(users_df, on=user_key, how="left")
else:
    joined = joined.join(
        users_df,
        F.col(user_key) == F.col(users_id_key),
        how="left"
    ).drop(users_id_key)
print(f"  After     ⟕ users  : {joined.count():,} rows")

# 3. Resultado ⟕ MCC
if mcc_key_txn:
    joined = joined.withColumn(mcc_key_txn, F.col(mcc_key_txn).cast("string"))
    mcc_renamed = mcc_df
    for c in mcc_renamed.columns:
        if c != "mcc_code" and c in joined.columns:
            mcc_renamed = mcc_renamed.withColumnRenamed(c, f"mcc_{c}")

    if mcc_key_txn != "mcc_code":
        joined = joined.join(
            mcc_renamed,
            F.col(mcc_key_txn) == F.col("mcc_code"),
            how="left"
        )
    else:
        joined = joined.join(mcc_renamed, on="mcc_code", how="left")
    print(f"  After     ⟕ mcc    : {joined.count():,} rows")

# 4. Resultado ⟕ etiquetas de fraude
txn_id_col = next(
    (c for c in joined.columns if c in ("transaction_id", "id", "trans_id")), None
)
if txn_id_col:
    joined = joined.withColumn(txn_id_col, F.col(txn_id_col).cast("string"))
    if txn_id_col != "transaction_id":
        labels_df2 = labels_df.withColumnRenamed("transaction_id", txn_id_col)
        joined = joined.join(labels_df2, on=txn_id_col, how="left")
    else:
        joined = joined.join(labels_df, on="transaction_id", how="left")
else:
    joined = joined.withColumn("is_fraud", F.lit(0))

print(f"  After     ⟕ labels  : {joined.count():,} rows")
print(f"  Final columns: {joined.columns}")


STEP 4 — Joining DataFrames
  Join: transactions ⟕ cards   on card_id
        result       ⟕ users   on client_id == user_id
        result       ⟕ mcc     on mcc
  After txn ⟕ cards  : 13,305,915 rows
  After     ⟕ users  : 13,305,915 rows
  After     ⟕ mcc    : 13,305,915 rows
  After     ⟕ labels  : 13,305,915 rows
  Final columns: ['id', 'card_id', 'date', 'client_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'card_client_id', 'card_brand', 'card_type', 'card_number', 'expires', 'cvv', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date', 'year_pin_last_changed', 'card_on_dark_web', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender', 'address', 'latitude', 'longitude', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards', 'mcc_code', 'description', 'is_fraud']


## 5 — Feature Engineering

In [7]:
# Se importan las funciones de ventana y agregación necesarias para el feature engineering
from pyspark.sql import Window
from pyspark.sql.functions import (
    col, count, mean, stddev, when, hour, dayofweek,
    unix_timestamp, lit, avg
)

print("=" * 60)
print("STEP 5 — Feature Engineering")
print("=" * 60)

# Se vuelven a detectar los nombres de columna después de todos los joins,
# ya que pueden haber cambiado por renombramientos previos
j_cols    = joined.columns
amt_col   = next((c for c in j_cols if c == "amount"), None)
ts_col2   = next((c for c in j_cols if c in ("timestamp", "date", "trans_date", "transaction_date")), None)
uid_col   = next((c for c in j_cols if c in ("user_id", "client_id")), None)
limit_col = next((c for c in j_cols if "limit" in c.lower() or "credit_limit" in c.lower()), None)
mcc_col2  = next((c for c in j_cols if "mcc" in c.lower() and "fraud" not in c.lower() and "rate" not in c.lower()), None)
fraud_col = next((c for c in j_cols if c == "is_fraud"), None)

print(f"  amount col    : {amt_col}")
print(f"  timestamp col : {ts_col2}")
print(f"  user_id col   : {uid_col}")
print(f"  card limit col: {limit_col}")
print(f"  mcc col       : {mcc_col2}")
print(f"  fraud col     : {fraud_col}")

enriched = joined

# ── 5a. transaction_velocity ─────────────────────────────────────────────────
# Se cuenta cuántas transacciones realizó el mismo usuario en las últimas 3 horas.
# Una velocidad alta puede ser señal de fraude (uso masivo del número de tarjeta).
# Se usa una Window ordenada por tiempo con un rango de -3 horas a 0 (en segundos).
if uid_col and ts_col2:
    print("  Computing transaction_velocity ...")
    ts_unix = unix_timestamp(col(ts_col2)).cast("long")
    w_vel = (
        Window.partitionBy(uid_col)
        .orderBy(ts_unix)
        .rangeBetween(-3 * 3600, 0)   # 3 horas en segundos
    )
    enriched = enriched.withColumn("_ts_unix", ts_unix)
    enriched = enriched.withColumn(
        "transaction_velocity",
        count(lit(1)).over(w_vel).cast("double")
    )
else:
    enriched = enriched.withColumn("transaction_velocity", lit(1.0))
print("  ✓ transaction_velocity done")

# ── 5b. amount_deviation ──────────────────────────────────────────────────────
# Se calcula cuántas desviaciones estándar se aleja el monto de esta transacción
# respecto al promedio histórico del usuario. Fórmula: (x - media) / std.
# Un valor alto indica que la transacción es inusualmente grande para ese usuario.
if amt_col and uid_col:
    print("  Computing amount_deviation ...")
    w_user = Window.partitionBy(uid_col)
    enriched = (
        enriched
        .withColumn("_user_mean", mean(col(amt_col)).over(w_user))
        .withColumn("_user_std",  stddev(col(amt_col)).over(w_user))
        .withColumn(
            "amount_deviation",
            # Se evita dividir por cero: si std = 0, la desviación es 0
            when(col("_user_std") > 0,
                 (col(amt_col) - col("_user_mean")) / col("_user_std"))
            .otherwise(lit(0.0))
        )
    )
else:
    enriched = enriched.withColumn("amount_deviation", lit(0.0))
print("  ✓ amount_deviation done")

# ── 5c. is_weekend ────────────────────────────────────────────────────────────
# Se marca si la transacción ocurrió en fin de semana (sábado o domingo).
# En Spark, dayofweek devuelve 1=domingo, 7=sábado.
if ts_col2:
    enriched = enriched.withColumn(
        "is_weekend",
        when(dayofweek(col(ts_col2)).isin(1, 7), True).otherwise(False)
    )
else:
    enriched = enriched.withColumn("is_weekend", lit(False))
print("  ✓ is_weekend done")

# ── 5d. is_night ──────────────────────────────────────────────────────────────
# Se marca si la transacción ocurrió de noche (entre las 22:00 y las 06:00).
# Las transacciones nocturnas tienen mayor probabilidad de ser fraudulentas.
if ts_col2:
    enriched = enriched.withColumn(
        "is_night",
        when((hour(col(ts_col2)) >= 22) | (hour(col(ts_col2)) < 6), True).otherwise(False)
    )
else:
    enriched = enriched.withColumn("is_night", lit(False))
print("  ✓ is_night done")

# ── 5e. mcc_fraud_rate ────────────────────────────────────────────────────────
# Se calcula la tasa de fraude promedio para cada categoría de comercio (MCC).
# Esto permite que el modelo aprenda que ciertos tipos de comercios son más riesgosos.
if mcc_col2 and fraud_col:
    print("  Computing mcc_fraud_rate ...")
    w_mcc = Window.partitionBy(mcc_col2)
    enriched = enriched.withColumn(
        "mcc_fraud_rate",
        avg(col(fraud_col).cast("double")).over(w_mcc)
    )
else:
    enriched = enriched.withColumn("mcc_fraud_rate", lit(0.0))
print("  ✓ mcc_fraud_rate done")

# ── 5f. card_utilization ──────────────────────────────────────────────────────
# Se calcula qué proporción del límite de la tarjeta se usó en esta transacción.
# Un valor cercano o mayor a 1 puede indicar uso inusual o fraudulento.
if amt_col and limit_col:
    enriched = enriched.withColumn(
        "card_utilization",
        when(col(limit_col) > 0, col(amt_col) / col(limit_col)).otherwise(lit(0.0))
    )
else:
    enriched = enriched.withColumn("card_utilization", lit(0.0))
print("  ✓ card_utilization done")

# Se eliminan las columnas auxiliares temporales creadas durante los cálculos intermedios
helper_cols = ["_ts_unix", "_user_mean", "_user_std"]
helper_cols = [c for c in helper_cols if c in enriched.columns]
enriched = enriched.drop(*helper_cols)
enriched.cache()

print(f"\n  ✓ Feature engineering complete")
print(f"  Final column count : {len(enriched.columns)}")
enriched.printSchema()

STEP 5 — Feature Engineering
  amount col    : amount
  timestamp col : date
  user_id col   : client_id
  card limit col: credit_limit
  mcc col       : mcc
  fraud col     : is_fraud
  Computing transaction_velocity ...
  ✓ transaction_velocity done
  Computing amount_deviation ...
  ✓ amount_deviation done
  ✓ is_weekend done
  ✓ is_night done
  Computing mcc_fraud_rate ...
  ✓ mcc_fraud_rate done
  ✓ card_utilization done

  ✓ Feature engineering complete
  Final column count : 46
root
 |-- id: string (nullable = true)
 |-- card_id: integer (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- amount: float (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: double (nullable = true)
 |-- mcc: string (nullable = true)
 |-- errors: string (nullable = true)
 |-- card_client_id: integer (

## 6 — Aggregations

In [8]:
spark.conf.set("spark.sql.session.timeZone", "UTC")


In [9]:
from pyspark.sql.functions import sum as spark_sum, round as spark_round

print("=" * 60)
print("STEP 6 — Aggregations")
print("=" * 60)

fraud_col2  = next((c for c in enriched.columns if c == "is_fraud"), None)
age_col     = next((c for c in enriched.columns if "age" in c.lower()), None)
card_type_c = next((c for c in enriched.columns if "card_type" in c.lower()), None)
amt_col2    = next((c for c in enriched.columns if c == "amount"), None)
ts_col3     = next((c for c in enriched.columns if c in ("timestamp", "date", "trans_date", "transaction_date")), None)
mcc_col3    = next((c for c in enriched.columns if "mcc" in c.lower() and "fraud" not in c.lower() and "rate" not in c.lower()), None)

def write_df(df, base_dir, name, show_n=5):
    path = os.path.join(base_dir, name + ".csv")
    pdf = df.toPandas()
    pdf.to_csv(path, index=False)
    print(f"  ✓ {name} saved → {path}")
    print(pdf.head(show_n).to_string(index=False))


# ── 6a. Fraud rate by MCC ─────────────────────────────────────────────────────
if mcc_col3 and fraud_col2:
    fraud_by_mcc = (
        enriched.groupBy(mcc_col3)
        .agg(
            count("*").alias("total_transactions"),
            spark_sum(col(fraud_col2).cast("int")).alias("fraud_count"),
            spark_round(avg(col(fraud_col2).cast("double")), 4).cast("double").alias("fraud_rate")
        )
    )
    write_df(fraud_by_mcc, PROCESSED_DIR, "fraud_rate_by_mcc")

# ── 6b. Avg transaction amount by age group ───────────────────────────────────
if age_col and amt_col2:
    enriched_with_age_grp = enriched.withColumn(
        "age_group",
        when(col(age_col).cast("int") < 25, "18-24")
        .when(col(age_col).cast("int") < 35, "25-34")
        .when(col(age_col).cast("int") < 45, "35-44")
        .when(col(age_col).cast("int") < 55, "45-54")
        .when(col(age_col).cast("int") < 65, "55-64")
        .otherwise("65+")
    )
    avg_by_age = (
        enriched_with_age_grp.groupBy("age_group")
        .agg(
            spark_round(avg(col(amt_col2).cast("double")), 2).cast("double").alias("avg_amount")
        )
        .orderBy("age_group")
    )
    write_df(avg_by_age, PROCESSED_DIR, "avg_amount_by_age_group")

# ── 6c. Fraud count by hour and day of week ───────────────────────────────────
if ts_col3 and fraud_col2:
    fraud_by_time = (
        enriched
        .withColumn("hour_of_day", hour(col(ts_col3)))
        .withColumn("day_of_week", dayofweek(col(ts_col3)))
        .groupBy("hour_of_day", "day_of_week")
        .agg(spark_sum(col(fraud_col2).cast("int")).alias("fraud_count"))
        .orderBy("day_of_week", "hour_of_day")
    )
    write_df(fraud_by_time, PROCESSED_DIR, "fraud_count_by_time", show_n=10)

# ── 6d. Fraud rate by card type ───────────────────────────────────────────────
if card_type_c and fraud_col2:
    fraud_by_card = (
        enriched.groupBy(card_type_c)
        .agg(
            count("*").alias("total"),
            spark_sum(col(fraud_col2).cast("int")).alias("fraud_count"),
            spark_round(avg(col(fraud_col2).cast("double")), 4).cast("double").alias("fraud_rate")
        )
    )
    write_df(fraud_by_card, PROCESSED_DIR, "fraud_rate_by_card_type")

print("\n  ✓ All aggregations complete")


STEP 6 — Aggregations
  ✓ fraud_rate_by_mcc saved → D:/UP/BigData/Pryecto1/processed\fraud_rate_by_mcc.csv
 mcc  total_transactions  fraud_count  fraud_rate
3005                 391          NaN         NaN
3260                2809          NaN         NaN
3390                8135          NaN         NaN
3393                7967          NaN         NaN
3395                8128          NaN         NaN
  ✓ avg_amount_by_age_group saved → D:/UP/BigData/Pryecto1/processed\avg_amount_by_age_group.csv
age_group  avg_amount
    18-24       47.72
    25-34       47.08
    35-44       40.79
    45-54       43.49
    55-64       42.54
  ✓ fraud_count_by_time saved → D:/UP/BigData/Pryecto1/processed\fraud_count_by_time.csv
 hour_of_day  day_of_week  fraud_count
           0            1          NaN
           1            1          NaN
           2            1          NaN
           3            1          NaN
           4            1          NaN
           5            1          NaN
  

## 7 — Save Enriched DataFrame

In [10]:
spark.conf.set("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem")
spark.conf.set("spark.hadoop.fs.file.impl.disable.cache", "true")


In [11]:
print("=" * 60)
print("STEP 7 — Saving enriched transactions to parquet")
print("=" * 60)

output_path = os.path.join(PROCESSED_DIR, "transactions_enriched")

# Se reduce el número de particiones a 4 antes de guardar para obtener
# menos archivos en disco y facilitar la lectura posterior
enriched_out = enriched.coalesce(4)   # en lugar de repartition(4)


# Se guarda el DataFrame enriquecido en formato parquet, que es eficiente en columnas
# y compatible con herramientas como pandas, Spark y otros sistemas de big data
enriched_out.write.mode("overwrite").parquet(output_path)

print(f"  ✓ Enriched data saved to {output_path}")
print(f"  Total rows : {enriched.count():,}")
print(f"  Columns    : {enriched.columns}")

# Se vuelve a leer el archivo recién guardado para verificar que se escribió correctamente
verify = spark.read.parquet(output_path)
print(f"  Verify read: {verify.count():,} rows, {len(verify.columns)} cols")
verify.show(3, truncate=True)

STEP 7 — Saving enriched transactions to parquet
  ✓ Enriched data saved to D:/UP/BigData/Pryecto1/processed\transactions_enriched
  Total rows : 13,305,915
  Columns    : ['id', 'card_id', 'date', 'client_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'card_client_id', 'card_brand', 'card_type', 'card_number', 'expires', 'cvv', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date', 'year_pin_last_changed', 'card_on_dark_web', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender', 'address', 'latitude', 'longitude', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards', 'mcc_code', 'description', 'is_fraud', 'transaction_velocity', 'amount_deviation', 'is_weekend', 'is_night', 'mcc_fraud_rate', 'card_utilization']
  Verify read: 13,305,915 rows, 46 cols
+--------+-------+-------------------+---------+-------+-----------------+-----------+---------------+--------------+----

In [12]:
print("Stopping Spark session ...")
spark.stop()
print("✓ Spark session stopped. Batch processing complete.")

Stopping Spark session ...
✓ Spark session stopped. Batch processing complete.
